# EDA - `player_appearance_pass.csv`

Lean EDA of the pass event table - one row per pass attempt during the
tournament. **Not used to derive any column in the main panel**, so
every feature engineered below is genuinely new and answers RQ4 (does
pass data add to the goal-scoring model?).

Two deliverables:

1. **Data cleaning** - structural integrity, NULL semantics, domain bounds.
2. **Feature manifest** - which engineered candidates carry signal.

Section structure:

| ID | Section |
|----|---------|
| A | Data cleaning |
| B | Engineer candidate features |
| C | Bivariate signal vs `scored_after` (pooled + by position) |
| D | Redundancy + cluster-robust significance |
| E | Final manifest |


## Setup

In [1]:
"""Imports."""
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
"""Display options."""
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
pd.set_option("display.precision", 4)


In [3]:
"""Paths and load."""
DATA_DIR = PROJECT_ROOT / "data"
PASS_PATH = DATA_DIR / "player_appearance_pass.csv"
MAIN_PATH = DATA_DIR / "players_quarters_final.csv"

passes = pd.read_csv(PASS_PATH)
main = pd.read_csv(MAIN_PATH)

appearance_meta = (
    main[["player_appearance_id", "player_id", "fixture_id", "position",
          "is_home", "minute_in", "minute_out"]]
    .drop_duplicates(subset="player_appearance_id")
    .set_index("player_appearance_id")
)
passes.shape, main.shape


((29795, 7), (3486, 33))

In [4]:
"""Helpers (Wilson CI + binned-rate table)."""
from scipy import stats as _stats


def wilson_ci(successes: int, trials: int, alpha: float = 0.05) -> tuple[float, float]:
    if trials == 0:
        return float("nan"), float("nan")
    p = successes / trials
    z = _stats.norm.ppf(1.0 - alpha / 2.0)
    denom = 1.0 + (z ** 2) / trials
    centre = (p + (z ** 2) / (2.0 * trials)) / denom
    half = z * np.sqrt(p * (1.0 - p) / trials + (z ** 2) / (4.0 * trials ** 2)) / denom
    return float(centre - half), float(centre + half)


def binned_rate(df: pd.DataFrame, feature: str, target: str = "scored_after",
                bins=None, by: str | None = None) -> pd.DataFrame:
    work = df[[feature, target] + ([by] if by else [])].copy()
    if bins is not None:
        work["__bin"] = pd.cut(work[feature], bins=bins, include_lowest=True)
    else:
        work["__bin"] = work[feature]
    grouping = ["__bin"] if by is None else [by, "__bin"]
    g = work.groupby(grouping, observed=False, dropna=False)[target].agg(["size", "sum"])
    g = g.rename(columns={"size": "n", "sum": "n_pos"}).reset_index()
    g["rate"] = g["n_pos"] / g["n"].where(g["n"] > 0)
    ci = g.apply(lambda r: wilson_ci(int(r["n_pos"]), int(r["n"])), axis=1, result_type="expand")
    g["ci_lo"], g["ci_hi"] = ci[0], ci[1]
    return g.rename(columns={"__bin": "bin"})


PERIOD_ORDER = {"half_1": 0, "half_2": 1, "extra_time_1": 2, "extra_time_2": 3}
CHECKPOINTS = [
    ("H1_15", "half_1", 15), ("H1_30", "half_1", 30), ("H1_45", "half_1", 45),
    ("H2_15", "half_2", 15), ("H2_30", "half_2", 30), ("H2_45", "half_2", 45),
    ("ET1_15", "extra_time_1", 15),
]
CP_ORDER = {cp: PERIOD_ORDER[per] for cp, per, _ in CHECKPOINTS}
CP_MINUTE = {cp: m for cp, _, m in CHECKPOINTS}
CP_PERIOD = {cp: per for cp, per, _ in CHECKPOINTS}
MATCH_MINUTE = {"H1_15":15, "H1_30":30, "H1_45":45,
                "H2_15":60, "H2_30":75, "H2_45":90, "ET1_15":105}


## Section A - Data cleaning

| ID | Check |
|----|-------|
| A1 | Shape, dtypes, head |
| A2 | PK uniqueness on `id` |
| A3 | Missingness (`addressee_player_appearance_id` NULL = clearance / speculative long ball; `stage` may be empty string per the dictionary) |
| A4 | Domain values for `period`, `stage`, `accurate` |
| A5 | Stoppage-time encoding |
| A6 | Orphan appearances - both passer and addressee |
| A7 | Self-pass check (passer == addressee should not occur) |


### A1 - Shape & dtypes

In [5]:
print("shape:", passes.shape)
print()
print(passes.dtypes.to_string())
passes.head(3)


shape: (29795, 7)

id                                  int64
period                                str
player_appearance_id                int64
addressee_player_appearance_id    float64
accurate                             bool
minute                              int64
stage                                 str


,id,period,player_appearance_id,addressee_player_appearance_id,accurate,minute,stage
0,816920,half_1,40764,40770.0,True,27,middle
1,816921,half_1,40764,40770.0,True,40,middle
2,816922,half_2,40764,40770.0,True,4,bottom


### A2 - PK uniqueness

In [6]:
print(f"duplicate `id` rows: {passes['id'].duplicated().sum()}")
print(f"unique ids: {passes['id'].nunique()} / {len(passes)}")


duplicate `id` rows: 0
unique ids: 29795 / 29795


### A3 - Missingness

In [7]:
na = passes.isna().sum()
print("NaN counts per column:")
print(na.to_string())
print()
print(f"empty-string `stage`: {(passes['stage'] == '').sum()}")


NaN counts per column:
id                                 0
period                             0
player_appearance_id               0
addressee_player_appearance_id    41
accurate                           0
minute                             0
stage                             57

empty-string `stage`: 0


### A4 - Domain values

In [8]:
for c in ("period", "stage", "accurate"):
    vc = passes[c].value_counts(dropna=False)
    print(f"{c}:")
    print(vc.to_string())
    print()


period:
period
half_1          15708
half_2          13488
extra_time_1      308
extra_time_2      291

stage:
stage
middle    14369
bottom     8733
top        6636
NaN          57

accurate:
accurate
True     24961
False     4834



### A5 - Stoppage-time encoding

In [9]:
cap = {"half_1": 45, "half_2": 45, "extra_time_1": 15, "extra_time_2": 15}
passes["__cap"] = passes["period"].map(cap)
stoppage = passes[passes["minute"] > passes["__cap"]]
print(f"stoppage-time passes: {len(stoppage)} ({len(stoppage)/len(passes):.2%})")
print()
print("by period:")
print(stoppage.groupby("period")["minute"].agg(["count", "min", "max"]).to_string())
passes.drop(columns="__cap", inplace=True)


stoppage-time passes: 1826 (6.13%)

by period:
              count  min  max
period                       
extra_time_1     12   16   16
extra_time_2     42   16   18
half_1          683   46   50
half_2         1089   46   57


### A6 - Orphan appearances

Two FKs reference player_appearance.id: the passer (`player_appearance_id`)
and the addressee (`addressee_player_appearance_id`). Both should be
filtered against the main panel before any aggregation.


In [10]:
main_apps = set(main["player_appearance_id"].unique())
passer_apps = set(passes["player_appearance_id"].unique())
addressee_apps = set(passes["addressee_player_appearance_id"].dropna().astype(int).unique())

orphan_passer = passer_apps - main_apps
orphan_addr = addressee_apps - main_apps

print(f"distinct *passer* appearances: {len(passer_apps)}")
print(f"distinct *addressee* appearances: {len(addressee_apps)}")
print(f"distinct main appearances: {len(main_apps)}")
print(f"orphan passers:    {len(orphan_passer):3d}  ({passes['player_appearance_id'].isin(orphan_passer).sum()} rows)")
print(f"orphan addressees: {len(orphan_addr):3d}  ({passes['addressee_player_appearance_id'].isin(orphan_addr).sum()} rows)")


distinct *passer* appearances: 959
distinct *addressee* appearances: 968
distinct main appearances: 869
orphan passers:     91  (449 rows)


orphan addressees:  99  (461 rows)


### A7 - Self-pass check

In [11]:
self_pass = (passes["player_appearance_id"] == passes["addressee_player_appearance_id"]).sum()
print(f"self-pass rows (passer == addressee): {self_pass}")


self-pass rows (passer == addressee): 72


### Section A - findings (interpretation)

Decisions logged for Section B:

* Filter to passer in main; addressee orphans don't break aggregations
  (the receiver-side aggregate uses NULL-tolerant counts).
* Empty-string `stage` is treated as missing (no spatial info).
* All windowing rules from prior EDAs (strict cumul, strict last15)
  carry over.


## Section B - Engineered candidate features

11 candidates across volume, quality, spatial and tactical-role axes.

| # | Feature | Formula | Rationale |
|---|---------|---------|-----------|
| 1 | `cumul_passes`               | count | volume of passes attempted |
| 2 | `last15_passes`              | count | recent passing volume |
| 3 | `pass_accuracy`              | accurate / total | completion rate |
| 4 | `cumul_top_third_passes`     | count where stage=top | spatial volume |
| 5 | `top_third_pass_share`       | top_third / total | spatial concentration |
| 6 | `cumul_passes_received`      | count where this player is addressee | playmaker-target signal |
| 7 | `passes_received_share`      | received / (sent + received) | distributor vs target balance |
| 8 | `cumul_unspecified_passes`   | count where addressee is NULL | clearances / speculative long balls |
| 9 | `unspecified_pass_share`     | unspecified / total | how much of the player's passing is "long" / clearances |
| 10 | `last15_pass_uplift`        | last15 rate / cumul rate | RQ6 anchor for pass volume |
| 11 | `passes_per_minute_played`  | cumul / minutes_so_far | normalised passing load |


In [12]:
"""Filter and tag period_order."""
passes_sender = passes[passes["player_appearance_id"].isin(main_apps)].copy()
passes_sender["period_order"] = passes_sender["period"].map(PERIOD_ORDER)

passes_receiver = passes[passes["addressee_player_appearance_id"].notna()].copy()
passes_receiver["addressee_player_appearance_id"] = passes_receiver["addressee_player_appearance_id"].astype(int)
passes_receiver = passes_receiver[passes_receiver["addressee_player_appearance_id"].isin(main_apps)].copy()
passes_receiver["period_order"] = passes_receiver["period"].map(PERIOD_ORDER)

print(f"passes_sender:   {passes_sender.shape}")
print(f"passes_receiver: {passes_receiver.shape}")


passes_sender:   (29346, 8)
passes_receiver: (29293, 8)


In [13]:
"""Aggregate at checkpoint level."""
def _agg_sender(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("player_appearance_id")
    return pd.DataFrame({
        "passes": g.size(),
        "accurate": g.apply(lambda x: (x["accurate"] == True).sum()),
        "top_third": g.apply(lambda x: (x["stage"] == "top").sum()),
        "unspecified": g.apply(lambda x: x["addressee_player_appearance_id"].isna().sum()),
    })


def _agg_receiver(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("addressee_player_appearance_id")
    out = pd.DataFrame({"received": g.size()})
    out.index.name = "player_appearance_id"
    return out


def panel_for_cp(cp: str) -> pd.DataFrame:
    cp_per = CP_PERIOD[cp]
    cp_min = CP_MINUTE[cp]
    cp_ord = CP_ORDER[cp]

    cumul_s = (
        (passes_sender["period_order"] < cp_ord) |
        ((passes_sender["period_order"] == cp_ord) & (passes_sender["minute"] <= cp_min))
    )
    last15_s = (
        (passes_sender["period"] == cp_per) &
        (passes_sender["minute"] > cp_min - 15) &
        (passes_sender["minute"] <= cp_min)
    )
    cumul_r = (
        (passes_receiver["period_order"] < cp_ord) |
        ((passes_receiver["period_order"] == cp_ord) & (passes_receiver["minute"] <= cp_min))
    )
    last15_r = (
        (passes_receiver["period"] == cp_per) &
        (passes_receiver["minute"] > cp_min - 15) &
        (passes_receiver["minute"] <= cp_min)
    )

    cumul_send = _agg_sender(passes_sender[cumul_s]).add_prefix("cumul_")
    last15_send = _agg_sender(passes_sender[last15_s]).add_prefix("last15_")
    cumul_recv = _agg_receiver(passes_receiver[cumul_r]).add_prefix("cumul_")
    last15_recv = _agg_receiver(passes_receiver[last15_r]).add_prefix("last15_")

    out = (
        cumul_send.join(last15_send, how="outer")
        .join(cumul_recv, how="outer")
        .join(last15_recv, how="outer")
        .fillna(0)
    )
    out["checkpoint"] = cp
    return out.reset_index()


panel = pd.concat([panel_for_cp(cp) for cp, _, _ in CHECKPOINTS], ignore_index=True)
panel.shape


(5338, 12)

In [14]:
"""Compose the 11 candidate features."""
EPS = 1e-6
am = appearance_meta.reset_index()[["player_appearance_id", "minute_in", "minute_out"]]
panel = panel.merge(am, on="player_appearance_id", how="left")
panel["match_minute_at_cp"] = panel["checkpoint"].map(MATCH_MINUTE)
panel["mins_played_so_far"] = (
    panel[["match_minute_at_cp", "minute_out"]].min(axis=1)
    - panel["minute_in"].clip(lower=1) + 1
).clip(lower=1)

feat = pd.DataFrame({
    "player_appearance_id": panel["player_appearance_id"],
    "checkpoint": panel["checkpoint"],
})
feat["cumul_passes"]            = panel["cumul_passes"]
feat["last15_passes"]           = panel["last15_passes"]
feat["pass_accuracy"]           = panel["cumul_accurate"] / panel["cumul_passes"].replace(0, np.nan)
feat["cumul_top_third_passes"]  = panel["cumul_top_third"]
feat["top_third_pass_share"]    = panel["cumul_top_third"] / panel["cumul_passes"].replace(0, np.nan)
feat["cumul_passes_received"]   = panel["cumul_received"]
feat["passes_received_share"]   = panel["cumul_received"] / (
    panel["cumul_passes"] + panel["cumul_received"]
).replace(0, np.nan)
feat["cumul_unspecified_passes"] = panel["cumul_unspecified"]
feat["unspecified_pass_share"]   = panel["cumul_unspecified"] / panel["cumul_passes"].replace(0, np.nan)

last15_rate = panel["last15_passes"] / 15.0
cumul_rate  = panel["cumul_passes"] / panel["mins_played_so_far"]
feat["last15_pass_uplift"]      = last15_rate / (cumul_rate + EPS)
feat["passes_per_minute_played"] = panel["cumul_passes"] / panel["mins_played_so_far"]

feat = feat.replace([np.inf, -np.inf], np.nan)
feat["last15_pass_uplift"] = feat["last15_pass_uplift"].clip(upper=10)
feat.describe().T[["count", "mean", "std", "min", "50%", "max"]]


,count,mean,std,min,50%,max
player_appearance_id,5338.0,43422.9142,7454.5587,39421.0,41253.0000,65789.000
cumul_passes,5338.0,24.7124,20.3641,0.0,19.0000,118.000
last15_passes,5338.0,5.1424,5.1058,0.0,4.0000,36.000
pass_accuracy,5309.0,0.8235,0.1552,0.0,0.8571,1.000
cumul_top_third_passes,5338.0,5.1688,6.6101,0.0,3.0000,55.000
top_third_pass_share,5309.0,0.2483,0.2367,0.0,0.2000,1.000
cumul_passes_received,5338.0,24.6669,19.8549,0.0,19.0000,119.000
passes_received_share,5338.0,0.5087,0.0785,0.0,0.5000,1.000
cumul_unspecified_passes,5338.0,0.0378,0.2641,0.0,0.0000,4.000
unspecified_pass_share,5309.0,0.0019,0.0146,0.0,0.0000,0.375


### B - findings (interpretation)

Sanity checks:

* `cumul_passes` should grow with checkpoint - median values jump from
  `H1_15` (low single digits) up to `H2_30` (~20+ for active outfielders).
* `pass_accuracy` should land roughly in the 60-90% range typical of
  professional tournament football; very low values (<30%) flag GKs
  punting clearances counted as "passes".
* `passes_received_share` close to 0.5 = balanced participation;
  closer to 0 = pure passer/distributor; closer to 1 = ball-magnet
  receiver (target striker / playmaker focal point).
* `unspecified_pass_share` should be small for ball-playing midfielders
  and high for goalkeepers / centre-backs (clearances).
* `last15_pass_uplift` undefined when `cumul_passes == 0` - early-cp
  rows can carry the same NaN.


## Section C - Bivariate signal vs `scored_after`

In [15]:
"""Merge for analysis."""
m = main[["player_appearance_id", "checkpoint", "scored_after",
          "fixture_id", "position"]].merge(
    feat, on=["player_appearance_id", "checkpoint"], how="left"
)
BASELINE = m["scored_after"].mean()
print(f"baseline scored_after rate: {BASELINE:.4f}")


baseline scored_after rate: 0.0582


In [16]:
"""Pearson r vs target."""
NUMERIC = [c for c in feat.columns if c not in ("player_appearance_id", "checkpoint")]
rows = []
for c in NUMERIC:
    s = m[c].dropna()
    if s.std() == 0:
        rows.append({"feature": c, "n": len(s), "pearson_r": float("nan")})
        continue
    t = m.loc[s.index, "scored_after"]
    rows.append({"feature": c, "n": len(s), "pearson_r": float(np.corrcoef(s, t)[0, 1])})
pearson_df = pd.DataFrame(rows).sort_values(
    "pearson_r", key=lambda x: x.abs(), ascending=False
).reset_index(drop=True)
pearson_df


,feature,n,pearson_r
0,top_third_pass_share,3431,0.1151
1,cumul_passes,3457,-0.1015
2,cumul_passes_received,3457,-0.0892
3,passes_per_minute_played,3457,-0.0747
4,passes_received_share,3457,0.0714
5,last15_passes,3457,-0.0500
6,cumul_unspecified_passes,3457,-0.0270
7,last15_pass_uplift,3457,0.0214
8,unspecified_pass_share,3431,-0.0137
9,pass_accuracy,3431,-0.0100


In [17]:
"""Binned target rates."""
BINS = {
    "cumul_passes":             [-1, 0, 10, 25, 200],
    "last15_passes":            [-1, 0, 3, 8, 50],
    "pass_accuracy":            [-0.01, 0.0, 0.6, 0.85, 1.01],
    "cumul_top_third_passes":   [-1, 0, 3, 8, 100],
    "top_third_pass_share":     [-0.01, 0.0, 0.2, 0.4, 1.01],
    "cumul_passes_received":    [-1, 0, 5, 15, 200],
    "passes_received_share":    [-0.01, 0.0, 0.4, 0.6, 1.01],
    "cumul_unspecified_passes": [-1, 0, 1, 3, 50],
    "unspecified_pass_share":   [-0.01, 0.0, 0.1, 0.3, 1.01],
    "last15_pass_uplift":       [-1, 0, 0.5, 1.5, 11],
    "passes_per_minute_played": [-0.01, 0.0, 0.5, 1.0, 10],
}

frames = []
for f, edges in BINS.items():
    t = binned_rate(m, f, "scored_after", bins=edges)
    t["feature"] = f
    frames.append(t)
binned_df = pd.concat(frames, ignore_index=True)[
    ["feature", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]
]
binned_df


,feature,bin,n,n_pos,rate,ci_lo,ci_hi
0,cumul_passes,"(-1.001, 0.0]",26,0,0.0000,0.0000e+00,0.1287
1,cumul_passes,"(0.0, 10.0]",1139,110,0.0966,8.0753e-02,0.1151
2,cumul_passes,"(10.0, 25.0]",1253,61,0.0487,3.8085e-02,0.0620
3,cumul_passes,"(25.0, 200.0]",1039,29,0.0279,1.9503e-02,0.0398
4,cumul_passes,NaN,29,3,0.1034,3.5815e-02,0.2639
5,last15_passes,"(-1.001, 0.0]",86,4,0.0465,1.8234e-02,0.1136
6,last15_passes,"(0.0, 3.0]",892,63,0.0706,5.5591e-02,0.0893
7,last15_passes,"(3.0, 8.0]",1441,86,0.0597,4.8581e-02,0.0731
8,last15_passes,"(8.0, 50.0]",1038,47,0.0453,3.4220e-02,0.0597
9,last15_passes,NaN,29,3,0.1034,3.5815e-02,0.2639


In [18]:
"""Stratified by position - top 3 features by |r| only."""
top3 = pearson_df.head(3)["feature"].tolist()
print(f"stratifying: {top3}")
sub = m[m["position"].isin(["A", "M", "D"])].copy()
strat_frames = []
for f in top3:
    t = binned_rate(sub, f, "scored_after", bins=BINS[f], by="position")
    t["feature"] = f
    strat_frames.append(t)
strat_df = pd.concat(strat_frames, ignore_index=True)[
    ["feature", "position", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]
]
strat_df


stratifying: ['top_third_pass_share', 'cumul_passes', 'cumul_passes_received']


,feature,position,bin,n,n_pos,rate,ci_lo,ci_hi
0,top_third_pass_share,A,"(-0.011, 0.0]",71,16,0.2254,1.4375e-01,0.3352
1,top_third_pass_share,A,"(0.0, 0.2]",50,4,0.0800,3.1550e-02,0.1884
2,top_third_pass_share,A,"(0.2, 0.4]",163,10,0.0613,3.3662e-02,0.1092
3,top_third_pass_share,A,"(0.4, 1.01]",305,44,0.1443,1.0925e-01,0.1881
4,top_third_pass_share,A,NaN,26,3,0.1154,4.0032e-02,0.2898
5,top_third_pass_share,D,"(-0.011, 0.0]",433,27,0.0624,4.3205e-02,0.0892
6,top_third_pass_share,D,"(0.0, 0.2]",515,6,0.0117,5.3502e-03,0.0252
7,top_third_pass_share,D,"(0.2, 0.4]",217,1,0.0046,8.1394e-04,0.0256
8,top_third_pass_share,D,"(0.4, 1.01]",75,4,0.0533,2.0933e-02,0.1293
9,top_third_pass_share,D,NaN,6,0,0.0000,2.7756e-17,0.3903


### C - findings (interpretation)

Patterns to watch:

* **`top_third_pass_share`** - mirrors the strongest signal across all
  prior EDAs (top-third concentration). Expect it to be the single
  strongest pass feature.
* **`passes_received_share`** - target strikers receive more than they
  send; expected positive monotone direction.
* **`pass_accuracy`** - direction unclear: high accuracy could be
  GKs/CBs (low scoring) OR midfielders (any scoring). Position
  stratification will resolve.
* **Volume features (`cumul_passes`)** - flat / weak based on the
  cross-EDA pattern that volume is uninformative.
* **`unspecified_pass_share`** - high for GKs/CBs (clearances) -
  expect a *negative* relationship.
* **`last15_pass_uplift`** - third-domain test of RQ6; uplifts have
  consistently failed in runs and pressure, expect the same here.


## Section D - Redundancy + cluster-robust significance

In [19]:
"""Pearson correlation matrix."""
corr = m[NUMERIC].corr(method="pearson")
hi = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .rename("pearson")
        .reset_index()
        .rename(columns={"level_0": "a", "level_1": "b"})
)
hi["abs"] = hi["pearson"].abs()
hi.sort_values("abs", ascending=False).head(15).reset_index(drop=True)


,a,b,pearson,abs
0,cumul_passes,cumul_passes_received,0.9829,0.9829
1,last15_passes,passes_per_minute_played,0.8421,0.8421
2,cumul_unspecified_passes,unspecified_pass_share,0.7813,0.7813
3,cumul_passes,passes_per_minute_played,0.6799,0.6799
4,cumul_passes_received,passes_per_minute_played,0.6559,0.6559
5,cumul_passes,last15_passes,0.5213,0.5213
6,cumul_top_third_passes,top_third_pass_share,0.5175,0.5175
7,last15_passes,cumul_passes_received,0.5002,0.5002
8,cumul_top_third_passes,cumul_passes_received,0.4725,0.4725
9,cumul_passes,cumul_top_third_passes,0.4685,0.4685


In [20]:
"""Cluster-robust GLM + BH-FDR."""
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


def cluster_robust_p(df: pd.DataFrame, feature: str) -> tuple[float, float, int]:
    sub = df[[feature, "scored_after", "fixture_id"]].dropna()
    if sub[feature].std() == 0 or len(sub) < 30:
        return float("nan"), float("nan"), len(sub)
    try:
        m_ = smf.logit(f"scored_after ~ {feature}", data=sub).fit(
            disp=False, maxiter=200,
            cov_type="cluster", cov_kwds={"groups": sub["fixture_id"]},
        )
        return float(m_.params[feature]), float(m_.pvalues[feature]), len(sub)
    except Exception:
        return float("nan"), float("nan"), len(sub)


rows = []
for f in NUMERIC:
    coef, p, n = cluster_robust_p(m, f)
    r = m[[f, "scored_after"]].corr().iloc[0, 1]
    rows.append({"feature": f, "n": n, "pearson_r": r, "coef": coef, "p_raw": p})
sig = pd.DataFrame(rows)

mask = sig["p_raw"].notna()
adj = np.full(len(sig), np.nan)
if mask.sum():
    _, adj_p, _, _ = multipletests(sig.loc[mask, "p_raw"].values, method="fdr_bh")
    adj[mask.values] = adj_p
sig["p_bh"] = adj
sig["bh_q05"] = sig["p_bh"] < 0.05
sig.sort_values("p_bh").reset_index(drop=True)


,feature,n,pearson_r,coef,p_raw,p_bh,bh_q05
0,top_third_pass_share,3431,0.1151,1.7319,2.1563e-05,0.0002,True
1,cumul_passes,3457,-0.1015,-0.0360,3.0110e-04,0.0017,True
2,cumul_passes_received,3457,-0.0892,-0.0310,8.2686e-04,0.0023,True
3,passes_received_share,3457,0.0714,3.0219,6.3297e-04,0.0023,True
4,passes_per_minute_played,3457,-0.0747,-1.2469,1.2916e-02,0.0284,True
5,last15_passes,3457,-0.0500,-0.0489,3.8932e-02,0.0634,False
6,cumul_unspecified_passes,3457,-0.0270,-1.3138,4.0368e-02,0.0634,False
7,last15_pass_uplift,3457,0.0214,0.2361,1.9817e-01,0.2725,False
8,unspecified_pass_share,3431,-0.0137,-5.6023,4.9186e-01,0.6012,False
9,pass_accuracy,3431,-0.0100,-0.2501,7.1960e-01,0.7916,False


### D - findings (interpretation)

Decisions:

* Pairs with `|pearson| >= 0.95` -> drop the dependent partner.
* BH q=0.05 survivors with rate above baseline -> KEEP.
* Negative-coef features with plausible mechanism (e.g.
  `unspecified_pass_share`) are kept - direction matters.


## Section E - Final manifest

In [21]:
"""Programmatic manifest."""
def best_non_baseline_bin(df: pd.DataFrame, feature: str) -> tuple[float, float, float, int]:
    sub = df[df["feature"] == feature]
    if sub.empty:
        return (float("nan"),) * 3 + (0,)
    sub = sub.sort_values("rate", ascending=False).iloc[0]
    return float(sub["rate"]), float(sub["ci_lo"]), float(sub["ci_hi"]), int(sub["n"])


rows = []
for feat_name in NUMERIC:
    rate, lo, hi, n = best_non_baseline_bin(binned_df, feat_name)
    rows.append({"feature": feat_name, "best_rate": rate,
                 "ci_lo": lo, "ci_hi": hi, "n": n})

manifest = pd.DataFrame(rows).merge(
    sig[["feature", "pearson_r", "p_bh", "bh_q05"]], on="feature", how="left"
)


def verdict(row) -> str:
    if pd.isna(row["best_rate"]):
        return "drop (no data)"
    rate_above = row["best_rate"] > BASELINE * 1.2
    ci_above = (not pd.isna(row["ci_lo"]) and row["ci_lo"] > BASELINE)
    if row["bh_q05"] is True and (rate_above or ci_above):
        return "KEEP"
    if ci_above:
        return "keep (cautious)"
    if row["best_rate"] > BASELINE * 1.4 and row["n"] >= 30:
        return "keep (cautious)"
    return "drop (flat)"


manifest["verdict"] = manifest.apply(verdict, axis=1)
manifest.sort_values(["verdict", "p_bh"]).reset_index(drop=True)


,feature,best_rate,ci_lo,ci_hi,n,pearson_r,p_bh,bh_q05,verdict
0,top_third_pass_share,0.1135,0.0926,0.1384,740,0.1151,0.0002,True,KEEP
1,cumul_passes,0.1034,0.0358,0.2639,29,-0.1015,0.0017,True,KEEP
2,cumul_passes_received,0.1034,0.0358,0.2639,29,-0.0892,0.0023,True,KEEP
3,passes_received_share,0.1555,0.1149,0.2069,238,0.0714,0.0023,True,KEEP
4,passes_per_minute_played,0.1034,0.0358,0.2639,29,-0.0747,0.0284,True,KEEP
5,last15_passes,0.1034,0.0358,0.2639,29,-0.0500,0.0634,False,drop (flat)
6,cumul_unspecified_passes,0.1034,0.0358,0.2639,29,-0.0270,0.0634,False,drop (flat)
7,last15_pass_uplift,0.1034,0.0358,0.2639,29,0.0214,0.2725,False,drop (flat)
8,unspecified_pass_share,0.0593,0.0518,0.0678,3357,-0.0137,0.6012,False,drop (flat)
9,pass_accuracy,0.0729,0.0482,0.1089,288,-0.0100,0.7916,False,drop (flat)


### Closing notes

* The manifest above is the deliverable. Once finalised, add a
  `features/passes.py` module with a single
  `build_pass_features(passes_df, main_df) -> DataFrame` entry,
  mirroring `features/shots.py`.
* Compare survivor count + rates against runs / pressure / shots EDAs
  to triage RQ4 ("does pass data add value?"). If pass features
  contribute >= what runs / pressure already deliver, the pass channel
  is justified.
* Strong pass features (especially `top_third_pass_share`,
  `passes_received_share`, `pass_accuracy x position`) become candidates
  for a fourth `build_pass_cross_features` entry in `features/cross.py`.
